# Single-Name CDS Swaptions in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Transaction types |
| 4 | Portfolio and transactions |
| 5 | Valuation |
| 6 | Instrument events |

## The instrument

A single-name CDS swaption gives you the right to enter into a credit default swap on one
specific reference entity, at a fixed spread, on a single expiry date. In LUSID that's a
`CdsOption` whose `underlying` points at a `CreditDefaultSwap`:

    underlying   the reference entity's own CDS, as a CreditDefaultSwap
    option       the CdsOption itself -- strike, expiry, option type

`CdsOption` is quote-driven by design -- it has no coupon and no projected interim cashflow of
its own. That means under `SimpleStatic` the underlying's economics never feed into the option's
PV; only the option's own quoted mark does.

## Constraint: create the underlying first

The underlying has to exist as its own upserted instrument before the option can reference it.
`CdsOption.underlying` takes a `MasteredInstrument` pointing at the underlying's LUID, not an
inline `CreditDefaultSwap` definition, so upsert the CDS first and then wrap its LUID in
`mastered()`.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

`underlying_version` pins the `CdsOption` to a specific AsAt version of the underlying. It has to
resolve to a version that actually exists, so we set it to "now" -- just after the underlying
gets upserted below -- rather than to any trade or economic date.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "SwaptionSingleNameCdsDemo"
RECIPE    = "swaption-single-name-cds-demo-recipe"
PORTFOLIO = "swaption-single-name-cds-demo-book"

CDS_ID    = "DEMO-NWIND-CDS"
CDS_DESC  = "Demo Northwind Industries CDS"
OPT_DESC  = "Demo Northwind Industries CDS Payer Swaption Dec-25"
CURRENCY  = "USD"

CDS_START    = d(2025, 6, 20)
CDS_MATURITY = d(2030, 6, 20)          # 5Y tenor off the CDS start

CDS_COUPON     = 0.0100                # 100bp fixed coupon
CDS_FREQUENCY  = "3M"
CDS_DAY_COUNT  = "Actual360"
CDS_NOTIONAL   = 1.0                   # per-unit underlying; the option's own notional carries size

TRADE_DATE  = d(2025, 6, 20)           # option trade date
EXPIRY_DATE = d(2025, 12, 22)          # before CDS_MATURITY
ASOF        = d(2025, 11, 14)          # before EXPIRY_DATE

STRIKE       = 0.0100                  # strike spread, 100bp
OPTION_TYPE  = "Payer"
DELIVERY     = "Cash"
EXERCISE     = "European"
OPT_NOTIONAL = 1.0                     # per-unit option; QUANTITY below carries the traded size

QUANTITY = 10_000_000.00
PRICE    = 0.65                        # points, quoted mark
DENOM    = 100

print(f"{OPT_DESC}")
print(f"  underlying {CDS_DESC} ({CDS_ID}) at {CDS_COUPON:.2%}, {CDS_FREQUENCY} {CDS_DAY_COUNT}")
print(f"  {OPTION_TYPE} swaption, strike {STRIKE:.2%}, expiry {EXPIRY_DATE:%Y-%m-%d}")
print(f"  {QUANTITY:,.0f} notional at {PRICE} points on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE} / {DENOM} = {QUANTITY * PRICE / DENOM:,.2f} {CURRENCY}")

Demo Northwind Industries CDS Payer Swaption Dec-25
  underlying Demo Northwind Industries CDS (DEMO-NWIND-CDS) at 1.00%, 3M Actual360
  Payer swaption, strike 1.00%, expiry 2025-12-22
  10,000,000 notional at 0.65 points on 2025-11-14
  market value = 10,000,000 x 0.65 / 100 = 65,000.00 USD


---
# 1. Instrument creation

We upsert the underlying first as its own `CreditDefaultSwap`, then reference it from the
`CdsOption` via `mastered()`.

In [3]:
cds = m.CreditDefaultSwap(
    instrument_type="CreditDefaultSwap",
    ticker=CDS_ID,
    start_date=CDS_START,
    maturity_date=CDS_MATURITY,
    coupon_rate=CDS_COUPON,
    notional=CDS_NOTIONAL,
    flow_conventions=m.CdsFlowConventions(
        currency=CURRENCY,
        payment_frequency=CDS_FREQUENCY,
        day_count_convention=CDS_DAY_COUNT,
        roll_convention="20",
        payment_calendars=[], reset_calendars=[]))

CDS_LUID = upsert("cds", CDS_DESC, CDS_ID, cds)
print(f"Underlying CDS : {CDS_LUID}")

swaption = m.CdsOption(
    instrument_type="CdsOption",
    start_date=TRADE_DATE,
    dom_ccy=CURRENCY,
    strike=STRIKE,
    delivery_type=DELIVERY,
    exercise_type=EXERCISE,
    notional=OPT_NOTIONAL,
    option_maturity_date=EXPIRY_DATE,
    option_type=OPTION_TYPE,
    underlying=mastered(CDS_LUID),
    underlying_version=datetime.now(timezone.utc))

OPT_LUID = upsert("swaption", OPT_DESC, "DEMO-NWIND-CDS-SWPN", swaption)
print(f"Swaption       : {OPT_LUID}")

Underlying CDS : LUID_00003DFZ


Swaption       : LUID_00003DG3


---
# 2. Recipe

Under `SimpleStatic`, the position is reported at a quoted mark. The underlying's conventions
just describe what the option refers to -- under this model, they don't feed into the number.

In [4]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Single-name CDS swaption, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="CdsOption")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: SwaptionSingleNameCdsDemo/swaption-single-name-cds-demo-recipe


---
# 3. Transaction types

`CdsOption` has two applicable events here: `ExpiryEvent` (the option lapses unexercised) and
`OptionExerciseCashEvent`, the cash-delivery counterpart to `OptionExercisePhysicalEvent` -- we
use the cash version since `DELIVERY` is `"Cash"`. Like other LUSID instrument events, both come
back from `query_applicable_instrument_events` with an empty `.transactions` list until we
register a matching transaction type -- see section 6.

`ExpiryEvent` maps to a transaction type named `Expiry`.`OptionExerciseCashEvent` maps to `OptionExerciseCash`, which we
register below for completeness. Section 6 shows this event never actually populates
`.transactions`, because its `supportedParticipationTypes` is `Voluntary` with a required
`OptionExerciseElection`, and `QueryApplicableInstrumentEventsRequest` has no field to supply one.

In [5]:
txn_config_api = api(lusid.TransactionConfigurationApi)

TXN_TYPES = [
    ("Expiry",             "Expire the swaption unexercised", -1),
    ("OptionExerciseCash", "Cash-exercise the swaption",      -1),
]

for txn_type, description, direction in TXN_TYPES:
    txn_config_api.set_transaction_type(
        source="default", type=txn_type, scope="default",
        transaction_type_request=m.TransactionTypeRequest(
            aliases=[m.TransactionTypeAlias(
                type=txn_type, description=description,
                transaction_class="Basic", transaction_roles="AllRoles", is_default=False)],
            movements=[m.TransactionTypeMovement(
                movement_types="StockMovement", side="Side1", direction=direction)]))
    print(f"{txn_type:<20} StockMovement Side1 {direction:+d}")

Expiry               StockMovement Side1 -1


OptionExerciseCash   StockMovement Side1 -1


---
# 4. Portfolio and transactions

`instrumentEventConfiguration` tells LUSID which recipe to use when forecasting this book's
events. It can only be set at portfolio creation time, so we pass `recipe=RECIPE` into
`recreate_portfolio()` below. Skip it and section 6 quietly returns zero events.

The swaption is bought for its spot premium. The transaction's own price is left at zero, because
the premium gets captured through the quoted mark at valuation rather than at trade booking.

In [6]:
recreate_portfolio(PORTFOLIO, "Single-Name CDS Swaption Demo Book", CURRENCY, d(2025, 1, 1),
                    recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-SWAPTION",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": OPT_LUID},
        transaction_date=TRADE_DATE.isoformat(),
        settlement_date=TRADE_DATE.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, TRADE_DATE, TRADE_DATE))

Recreated SwaptionSingleNameCdsDemo/swaption-single-name-cds-demo-book


,date,type,luid,units,consideration
0,2025-06-20,Buy,LUID_00003DG3,"10,000,000.00",0.00


---
# 5. Valuation

Just one quote, at the swaption's own price, expressed per unit.

In [7]:
upsert_price(OPT_LUID, PRICE / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == OPT_DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV)
0,Demo Northwind Industries CDS Payer Swaption D...,"10,000,000.00","65,000.00"


LUSID CleanPV 65,000.00  vs  quoted mark 65,000.00


---
# 6. Instrument events

Nothing below actually posts an event -- `query_applicable_instrument_events` just forecasts what
`option_maturity_date`, `exercise_type`, and `delivery_type` imply, over a window spanning the
swaption's own expiry.

In [8]:
events_api      = api(lusid.InstrumentEventsApi)
event_types_api = api(lusid.InstrumentEventTypesApi)

WINDOW_END = EXPIRY_DATE + timedelta(days=45)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=TRADE_DATE.isoformat(),
        window_end=WINDOW_END.isoformat(),
        effective_at=WINDOW_END.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

print(f"{len(applicable)} applicable event(s) over {TRADE_DATE:%Y-%m-%d} .. {WINDOW_END:%Y-%m-%d}:\n")
display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

2 applicable event(s) over 2025-06-20 .. 2026-02-05:



,event type,eligible balance,status
0,OptionExerciseCashEvent,"10,000,000.00",Active
1,ExpiryEvent,"10,000,000.00",Active


## 6a. Template specifications

`get_transaction_template_specification` returns `supportedParticipationTypes` and
`supportedElectionTypes` -- these are the fields that tell you which events can actually produce
a forecast transaction.

In [9]:
for event_type in ("ExpiryEvent", "OptionExerciseCashEvent"):
    sd = json.loads(event_types_api.get_transaction_template_specification(
        instrument_event_type=event_type).to_json())
    elections = [e.get("electionType") for e in (sd.get("supportedElectionTypes") or [])]
    print(event_type)
    print(f"  applies to CdsOption : {'CdsOption' in (sd.get('supportedInstrumentTypes') or [])}")
    print(f"  participation        : {sd.get('supportedParticipationTypes')}")
    print(f"  elections required   : {elections or 'none'}")
    print()

ExpiryEvent
  applies to CdsOption : True
  participation        : ['Mandatory']
  elections required   : none



OptionExerciseCashEvent
  applies to CdsOption : True
  participation        : ['Voluntary']
  elections required   : ['OptionExerciseElection']



## 6b. The transactions each event would book

`ExpiryEvent` is `Mandatory` -- once `Expiry` is registered in section 3, LUSID has everything it
needs and books a transaction. `OptionExerciseCashEvent` is `Voluntary` with a required
`OptionExerciseElection`, and `query_applicable_instrument_events` has no parameter to supply
one, so it can never produce a forecast transaction here, no matter what gets registered in
section 3. That's just how `Voluntary` participation works generally: it shows up as a forecast,
but never with transactions.

In [10]:
rows = []
for ev in applicable:
    for txn in (ev.transactions or []):
        rows.append({
            "event":    ev.instrument_event_type,
            "txn type": getattr(txn, "type", None),
            "units":    getattr(txn, "units", None),
        })

if rows:
    display(pd.DataFrame(rows))
else:
    print("No forecast transactions.")

print()
for ev in applicable:
    n = len(ev.transactions or [])
    print(f"{ev.instrument_event_type:<24} {n} transaction(s)")

,event,txn type,units
0,ExpiryEvent,Expiry,"10,000,000.00"



OptionExerciseCashEvent  0 transaction(s)
ExpiryEvent              1 transaction(s)


---
# Summary

1. A single-name CDS swaption is a `CdsOption` whose `underlying` points at the reference
   entity's own `CreditDefaultSwap`, mastered and upserted first.
2. `CdsOption.underlying` takes a `MasteredInstrument` pointing at the underlying's LUID -- you
   can't define the underlying inline.
3. It's reported at a quoted mark under `SimpleStatic`; the underlying's conventions describe
   what the option refers to, not the number itself.
4. `query_applicable_instrument_events` returns `ExpiryEvent` and `OptionExerciseCashEvent` for
   this cash-delivered swaption, both with an empty `.transactions` list until a matching
   transaction type gets registered. The type name LUSID looks for is the event type name with
   the `Event` suffix stripped -- `Expiry`, not an instrument-flavoured name like
   `CdsOptionExpiry`.
5. That rule only gets `ExpiryEvent` as far as a forecast transaction. `OptionExerciseCashEvent`
   stays empty no matter what gets registered, because it's `Voluntary` and needs an
   `OptionExerciseElection` that this query has no way to supply -- a platform limitation, not a
   naming issue.

In [11]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Underlying : {CDS_LUID}")
print(f"Swaption   : {OPT_LUID}")

Scope      : SwaptionSingleNameCdsDemo
Portfolio  : SwaptionSingleNameCdsDemo/swaption-single-name-cds-demo-book
Recipe     : SwaptionSingleNameCdsDemo/swaption-single-name-cds-demo-recipe
Underlying : LUID_00003DFZ
Swaption   : LUID_00003DG3
